# Limpeza e preparação dos dados para PLN

Este notebook filtra jogos e avaliações, limpa os textos e cria representações por tokens para uso posterior em modelos. Execute a coleta primeiro e confirme a existência dos dois arquivos pickle em `_DadosBrutos`.

## Fluxo e arquivos

Execute as células de cima para baixo: instalação; leitura e filtros; limpeza textual; tokenização; normalização; stopwords; stemming e lematização; e comparação. As etapas compartilham DataFrames em memória; após reiniciar o kernel, é necessário refazer as etapas anteriores.

| Diretório | Conteúdo |
| --- | --- |
| `_DadosBrutos/` | Entradas `jogos_steam.pkl.gz` e `steam_reviews.pkl.gz`, preservadas por este notebook |
| `_DadosLimpos/` | Os mesmos nomes de arquivo, atualizados a cada etapa principal |
| `_DadosLimpos/stemming/` | Versão alternativa com radicais |
| `_DadosLimpos/lematizacao/` | Versão alternativa com lemas |

Os caminhos são relativos à raiz do repositório. Os arquivos pickle de saída são sobrescritos: se a execução parar no meio, os arquivos principais refletem a última etapa salva, e saídas alternativas podem ser de uma execução anterior. Os dados e suas cópias são carregados em memória; as colunas de tokens também aumentam o tamanho dos arquivos.

## 1. Instalação das dependências

`%pip` instala as bibliotecas no ambiente do kernel. Esta etapa pode precisar de acesso à internet. Se o ambiente solicitar reinicialização, reinicie o kernel antes de continuar.

As exportacoes usam pickle com gzip (`.pkl.gz`, nivel 1). Os metadados de controle da coleta continuam em JSON.

Para reduzir o uso de memoria, tokens com o mesmo texto compartilham strings. A gravacao usa arquivo temporario e so substitui o pickle anterior depois de concluir. Em caso de MemoryError anterior, reinicie o kernel e execute desde a leitura para regenerar os arquivos.


In [18]:
%pip install pandas nltk langdetect simplemma


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Leitura e remoção de descrições vazias

Lê os arquivos pickle brutos e identifica jogos com `description` ausente ou composta apenas por espaços. Remove esses jogos e suas avaliações por `appid`, mantendo a coerência desse filtro entre os dois conjuntos.

São necessários, ao longo do notebook, `appid`, `name` e `description` no catálogo e `appid` e `review` nas avaliações. Os primeiros resultados são gravados em `_DadosLimpos`, e as contagens mostram o impacto do filtro.


In [19]:
from pathlib import Path

# Resolve o repositorio a partir da raiz ou de qualquer subpasta.
raiz_projeto = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "1_Coleta_Dados").is_dir()
     and (p / "2_Limpeza_Preparacao").is_dir()),
    None,
)
if raiz_projeto is None:
    raise RuntimeError("Execute o notebook dentro do repositorio PLN_SteamRecommend.")

import pandas as pd

import gc
import os
import sys
import tempfile


def salvar_pickle(dados, destino):
    # Compartilha strings repetidas sem alterar os tokens ou suas listas.
    # Reduz tambem a tabela de objetos que pickle precisa manter na memoria.
    for coluna in dados.columns:
        if '_tokens' in coluna:
            for tokens in dados[coluna]:
                if isinstance(tokens, list):
                    for i, token in enumerate(tokens):
                        tokens[i] = sys.intern(token)
    gc.collect()
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=destino.parent, suffix='.tmp', delete=False) as arquivo:
        temporario = Path(arquivo.name)
    try:
        dados.to_pickle(temporario, compression={'method': 'gzip', 'compresslevel': 1}, protocol=5)
        os.replace(temporario, destino)
    finally:
        temporario.unlink(missing_ok=True)




pasta_brutos = raiz_projeto / "_DadosBrutos"
pasta_limpos = raiz_projeto / "_DadosLimpos"
pasta_limpos.mkdir(parents=True, exist_ok=True)

caminho_jogos = pasta_brutos / "jogos_steam.pkl.gz"
caminho_reviews = pasta_brutos / "steam_reviews.pkl.gz"

jogos = pd.read_pickle(caminho_jogos)
reviews = pd.read_pickle(caminho_reviews)

descricao_vazia = jogos["description"].isna() | jogos["description"].astype("string").str.strip().eq("")
appids_removidos = set(jogos.loc[descricao_vazia, "appid"])

jogos_limpo = jogos.loc[~jogos["appid"].isin(appids_removidos)].copy()
reviews_limpo = reviews.loc[~reviews["appid"].isin(appids_removidos)].copy()

salvar_pickle(jogos_limpo, pasta_limpos / "jogos_steam.pkl.gz")
salvar_pickle(reviews_limpo, pasta_limpos / "steam_reviews.pkl.gz")

print(f"Jogos antes: {len(jogos):,}")
print(f"Reviews antes: {len(reviews):,}")
print(f"Jogos removidos por descricao vazia: {len(appids_removidos):,}")
print(f"Reviews removidas dos jogos sem descricao: {len(reviews) - len(reviews_limpo):,}")
print(f"Jogos depois: {len(jogos_limpo):,}")
print(f"Reviews depois: {len(reviews_limpo):,}")


Jogos antes: 188,027
Reviews antes: 558,623
Jogos removidos por descricao vazia: 341
Reviews removidas dos jogos sem descricao: 1,313
Jogos depois: 187,686
Reviews depois: 557,310


## 3. Limpeza dos textos e remocao de textos curtos

Extrai texto de HTML, ignora script e style, normaliza Unicode e espacos. Apos a limpeza, mantem descricoes e reviews com pelo menos **10 palavras**, antes da remocao de stopwords e da deteccao de idioma.

Ajuste `MIN_PALAVRAS_DESCRICAO` e `MIN_PALAVRAS_REVIEW` separadamente. A contagem considera palavras com letras, incluindo contracoes; numeros e pontuacao isolados nao contam. Textos vazios tambem sao excluidos.

Quando uma descricao e excluida, suas reviews tambem sao removidas. Uma review curta remove apenas a propria review. As colunas `description_n_palavras` e `review_n_palavras` registram as contagens nos arquivos pickle.

A pontuacao e removida apenas das descricoes dos jogos, substituida por espacos antes da tokenizacao. Nas reviews, a pontuacao permanece no texto e nos tokens para a futura analise de sentimentos. Os arquivos brutos permanecem preservados.


In [20]:
import re
import string
import unicodedata
from html.parser import HTMLParser


class ExtratorTextoHTML(HTMLParser):
    blocos = {
        "p", "div", "br", "hr", "li", "ul", "ol", "table", "tr", "td",
        "th", "h1", "h2", "h3", "h4", "h5", "h6", "section", "blockquote",
    }

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.partes = []
        self.ignorar = 0

    def handle_starttag(self, tag, attrs):
        if tag in {"script", "style"}:
            self.ignorar += 1
        if tag in self.blocos and not self.ignorar:
            self.partes.append(" ")

    def handle_endtag(self, tag):
        if tag in {"script", "style"}:
            self.ignorar = max(0, self.ignorar - 1)
        if tag in self.blocos and not self.ignorar:
            self.partes.append(" ")

    def handle_data(self, data):
        if not self.ignorar:
            self.partes.append(data)


def limpar_texto(texto):
    if pd.isna(texto):
        return ""
    parser = ExtratorTextoHTML()
    parser.feed(str(texto))
    parser.close()
    texto = unicodedata.normalize("NFKC", "".join(parser.partes))
    # Remove caracteres invisiveis sem retirar acentos, pontuacao ou negacoes.
    texto = texto.translate(dict.fromkeys(map(ord, "\u200b\ufeff\u00ad")))
    texto = "".join(
        c for c in texto
        if unicodedata.category(c) != "Cc" or c.isspace()
    )
    return re.sub(r"\s+", " ", texto).strip()


def limpar_descricao(texto):
    texto = limpar_texto(texto)
    # Substitui pontuacao por espacos para nao juntar palavras vizinhas.
    # Inclui pontuacao Unicode (aspas curvas, travessoes etc.) e ASCII.
    texto = ''.join(
        ' ' if unicodedata.category(c).startswith('P') or c in string.punctuation else c
        for c in texto
    )
    return re.sub(r'\s+', ' ', texto).strip()


jogos_limpo["description"] = jogos_limpo["description"].map(limpar_descricao)
reviews_limpo["review"] = reviews_limpo["review"].map(limpar_texto)

# Minimos independentes, contados antes de remover stopwords.
MIN_PALAVRAS_DESCRICAO = 10
MIN_PALAVRAS_REVIEW = 10


def contar_palavras(texto):
    # Conta palavras com letras; numeros e pontuacao isolados nao contam.
    # Contracoes como don't contam como uma palavra.
    return len(re.findall(r"[^\W\d_]+(?:['\u2019][^\W\d_]+)*", texto))


jogos_antes_tamanho = len(jogos_limpo)
reviews_antes_tamanho = len(reviews_limpo)
jogos_limpo['description_n_palavras'] = jogos_limpo['description'].map(contar_palavras)
jogos_limpo = jogos_limpo.loc[
    jogos_limpo['description_n_palavras'].ge(MIN_PALAVRAS_DESCRICAO)
].copy()

# Remove tambem as reviews de jogos excluidos ou ausentes do catalogo.
reviews_limpo = reviews_limpo.loc[
    reviews_limpo['appid'].isin(jogos_limpo['appid'])
].copy()
reviews_removidas_por_tamanho_jogo = reviews_antes_tamanho - len(reviews_limpo)
reviews_antes_tamanho_proprio = len(reviews_limpo)
reviews_limpo['review_n_palavras'] = reviews_limpo['review'].map(contar_palavras)
reviews_limpo = reviews_limpo.loc[
    reviews_limpo['review_n_palavras'].ge(MIN_PALAVRAS_REVIEW)
].copy()

salvar_pickle(jogos_limpo, pasta_limpos / 'jogos_steam.pkl.gz')
salvar_pickle(reviews_limpo, pasta_limpos / 'steam_reviews.pkl.gz')

print(f'Jogos removidos por descricao curta ou vazia: {jogos_antes_tamanho - len(jogos_limpo):,}')
print(f'Reviews removidas por jogo excluido ou ausente: {reviews_removidas_por_tamanho_jogo:,}')
print(f'Reviews removidas por texto curto ou vazio: {reviews_antes_tamanho_proprio - len(reviews_limpo):,}')
print(f'Jogos restantes: {len(jogos_limpo):,}')
print(f'Reviews restantes: {len(reviews_limpo):,}')


Jogos removidos por descricao curta ou vazia: 6,673
Reviews removidas por jogo excluido ou ausente: 15,505
Reviews removidas por texto curto ou vazio: 148,716
Jogos restantes: 181,013
Reviews restantes: 393,089


## 4. Manter apenas ingles

O idioma do jogo e determinado somente pela descricao limpa, sem filtrar pelo nome ou alfabeto. Mantemos apenas `description_idioma == "en"` e removemos as reviews de jogos excluidos. Nas reviews restantes, mantemos apenas `review_idioma == "en"`.

A deteccao usa ate 1.500 caracteres e exige probabilidade estimada de pelo menos 0,90. Textos com menos de cinco palavras ou 20 letras, resultados incertos e falhas de deteccao sao `indeterminado` e tambem sao removidos. Isso pode excluir textos curtos em ingles. A semente fixa torna a deteccao reproduzivel no mesmo ambiente.

As contagens por idioma sao mostradas antes da exclusao. As colunas de idioma sao preservadas nas exportacoes pickle e reutilizadas nas etapas seguintes.


In [21]:
from langdetect import DetectorFactory, detect_langs, LangDetectException
from IPython.display import display

DetectorFactory.seed = 0

def detectar_idioma(texto):
    if pd.isna(texto):
        return "indeterminado"
    # A detecao e heuristica; textos curtos e resultados incertos serao removidos.
    amostra = str(texto)[:1500]
    if len(amostra.split()) < 5 or sum(c.isalpha() for c in amostra) < 20:
        return "indeterminado"
    try:
        candidato = detect_langs(amostra)[0]
    except LangDetectException:
        return "indeterminado"
    return candidato.lang if candidato.prob >= 0.90 else "indeterminado"


jogos_antes_idioma = len(jogos_limpo)
reviews_antes_idioma = len(reviews_limpo)
jogos_limpo['description_idioma'] = jogos_limpo['description'].map(detectar_idioma)
print('Descricoes por idioma (antes do filtro):')
display(jogos_limpo['description_idioma'].value_counts(dropna=False))

# O nome do jogo nao participa do filtro.
jogos_limpo = jogos_limpo.loc[jogos_limpo['description_idioma'].eq('en')].copy()
reviews_limpo = reviews_limpo.loc[
    reviews_limpo['appid'].isin(jogos_limpo['appid'])
].copy()
reviews_removidas_por_jogo = reviews_antes_idioma - len(reviews_limpo)
reviews_antes_filtro_proprio = len(reviews_limpo)

reviews_limpo['review_idioma'] = reviews_limpo['review'].map(detectar_idioma)
print('Reviews de jogos mantidos, por idioma (antes do filtro):')
display(reviews_limpo['review_idioma'].value_counts(dropna=False))
reviews_limpo = reviews_limpo.loc[reviews_limpo['review_idioma'].eq('en')].copy()

salvar_pickle(jogos_limpo, pasta_limpos / 'jogos_steam.pkl.gz')
salvar_pickle(reviews_limpo, pasta_limpos / 'steam_reviews.pkl.gz')

print(f'Jogos removidos por idioma: {jogos_antes_idioma - len(jogos_limpo):,}')
print(f'Reviews removidas por jogo ausente ou excluido: {reviews_removidas_por_jogo:,}')
print(f'Reviews removidas pelo proprio idioma: {reviews_antes_filtro_proprio - len(reviews_limpo):,}')
print(f'Jogos em ingles restantes: {len(jogos_limpo):,}')
print(f'Reviews em ingles restantes: {len(reviews_limpo):,}')


Descricoes por idioma (antes do filtro):


description_idioma
en               178720
indeterminado      1177
zh-cn               497
ko                  167
ja                  150
fr                   49
es                   46
ru                   44
de                   39
pt                   36
tr                   19
pl                    9
it                    8
ar                    7
cs                    7
he                    4
vi                    4
th                    4
uk                    4
ro                    3
af                    3
no                    3
fi                    2
el                    2
nl                    2
ca                    2
hr                    1
da                    1
sk                    1
hu                    1
so                    1
Name: count, dtype: int64

Reviews de jogos mantidos, por idioma (antes do filtro):


review_idioma
en               379916
indeterminado      4679
ru                 2597
es                  688
pt                  519
de                  308
fr                  210
tr                  206
ar                  136
pl                  126
zh-cn               114
id                  112
vi                  109
ko                  107
th                   80
uk                   64
it                   60
ja                   52
ro                   48
no                   31
cs                   30
nl                   28
so                   25
fi                   19
af                   19
hu                   17
et                   17
cy                   14
tl                   13
sk                   12
fa                   12
da                   11
hr                    9
el                    9
he                    9
ca                    8
bg                    7
sv                    6
lt                    6
sl                    4
sw                    3
lv

Jogos removidos por idioma: 2,293
Reviews removidas por jogo ausente ou excluido: 2,642
Reviews removidas pelo proprio idioma: 10,531
Jogos em ingles restantes: 178,720
Reviews em ingles restantes: 379,916


## 5. Tokenização

Usa `TweetTokenizer` para separar os textos em tokens, preservando maiúsculas, repetições de caracteres e identificadores com `@`. Cria `description_tokens` e `review_tokens`.

As listas de tokens ficam preservadas no pickle. Use `pd.read_pickle` para recuperar o DataFrame com as listas, sem conversao JSON.


In [22]:
import sys
from nltk.tokenize import TweetTokenizer

tokenizador = TweetTokenizer(preserve_case=True, reduce_len=False, strip_handles=False)


def tokenizar_texto(texto):
    if pd.isna(texto):
        return []
    return [sys.intern(token) for token in tokenizador.tokenize(str(texto))]


jogos_limpo["description_tokens"] = jogos_limpo["description"].map(tokenizar_texto)
reviews_limpo["review_tokens"] = reviews_limpo["review"].map(tokenizar_texto)

# Pickle preserva as listas de tokens diretamente.
for dados, coluna_tokens, nome_arquivo in [
    (jogos_limpo, "description_tokens", "jogos_steam.pkl.gz"),
    (reviews_limpo, "review_tokens", "steam_reviews.pkl.gz"),
]:
    salvar_pickle(dados, pasta_limpos / nome_arquivo)

print(f"Tokens nas descricoes: {jogos_limpo['description_tokens'].map(len).sum():,}")
print(f"Tokens nas reviews: {reviews_limpo['review_tokens'].map(len).sum():,}")
print(f"Reviews sem tokens: {reviews_limpo['review_tokens'].map(len).eq(0).sum():,}")

display(jogos_limpo[["appid", "description", "description_tokens"]].head(3))
display(reviews_limpo[["appid", "review", "review_tokens"]].head(3))


Tokens nas descricoes: 6,468,410
Tokens nas reviews: 51,675,831
Reviews sem tokens: 0


,appid,description,description_tokens
0,10,Play the world s number 1 online action game E...,"[Play, the, world, s, number, 1, online, actio..."
1,20,One of the most popular online action games of...,"[One, of, the, most, popular, online, action, ..."
2,30,Enlist in an intense brand of Axis vs Allied t...,"[Enlist, in, an, intense, brand, of, Axis, vs,..."


,appid,review,review_tokens
0,10,"Good game, most perfect counter strike ever ex...","[Good, game, ,, most, perfect, counter, strike..."
2,10,This game is good for anyone who wants a smoot...,"[This, game, is, good, for, anyone, who, wants..."
8,10,Very good game. I like the older graphics and ...,"[Very, good, game, ., I, like, the, older, gra..."


## 6. Normalização dos tokens

Padroniza Unicode e variantes de apóstrofos e converte os tokens para minúsculas. Mantém as colunas anteriores e acrescenta `description_tokens_normalizados` e `review_tokens_normalizados`.

A gravação serializa ambas as versões de tokens em JSON, permitindo comparar a tokenização original com a representação normalizada.


In [23]:
import sys
import unicodedata

apostrofos = str.maketrans({"\u2018": "'", "\u2019": "'", "\u02bc": "'"})


def normalizar_tokens(tokens):
    return [
        sys.intern(unicodedata.normalize(
            "NFKC", unicodedata.normalize("NFKC", token).translate(apostrofos).lower()
        ))
        for token in tokens
    ]


jogos_limpo["description_tokens_normalizados"] = jogos_limpo["description_tokens"].map(
    normalizar_tokens
)
reviews_limpo["review_tokens_normalizados"] = reviews_limpo["review_tokens"].map(
    normalizar_tokens
)

# Pickle preserva as listas de tokens diretamente.
for dados, coluna_tokens, nome_arquivo in [
    (jogos_limpo, "description_tokens", "jogos_steam.pkl.gz"),
    (reviews_limpo, "review_tokens", "steam_reviews.pkl.gz"),
]:
    colunas_listas = [coluna_tokens, f"{coluna_tokens}_normalizados"]
    salvar_pickle(dados, pasta_limpos / nome_arquivo)

print(f"Descricoes normalizadas: {len(jogos_limpo):,}")
print(f"Reviews normalizadas: {len(reviews_limpo):,}")

display(jogos_limpo[
    ["appid", "description_tokens", "description_tokens_normalizados"]
].head(3))
display(reviews_limpo[
    ["appid", "review_tokens", "review_tokens_normalizados"]
].head(3))


Descricoes normalizadas: 178,720
Reviews normalizadas: 379,916


,appid,description_tokens,description_tokens_normalizados
0,10,"[Play, the, world, s, number, 1, online, actio...","[play, the, world, s, number, 1, online, actio..."
1,20,"[One, of, the, most, popular, online, action, ...","[one, of, the, most, popular, online, action, ..."
2,30,"[Enlist, in, an, intense, brand, of, Axis, vs,...","[enlist, in, an, intense, brand, of, axis, vs,..."


,appid,review_tokens,review_tokens_normalizados
0,10,"[Good, game, ,, most, perfect, counter, strike...","[good, game, ,, most, perfect, counter, strike..."
2,10,"[This, game, is, good, for, anyone, who, wants...","[this, game, is, good, for, anyone, who, wants..."
8,10,"[Very, good, game, ., I, like, the, older, gra...","[very, good, game, ., i, like, the, older, gra..."


## 7. Remocao de stopwords em ingles

Reutiliza os idiomas definidos pelo filtro anterior e usa a lista inglesa do NLTK. Preserva negacoes para reduzir alteracoes de sentido. Acrescenta as colunas `_tokens_sem_stopwords` e atualiza os arquivos pickle principais.


In [24]:
import nltk
from nltk.corpus import stopwords

try:
    stopwords.fileids()
except LookupError:
    nltk.download("stopwords", raise_on_error=True)

idiomas_stopwords = {"en": "english"}
# Preserva negacoes para nao inverter o sentido, especialmente nas reviews.
negacoes = set(normalizar_tokens([
    "no", "not", "nor", "never", "neither", "nothing", "nobody", "without",
    "cannot", "n't",
]))
stopwords_por_idioma = {
    codigo: {
        palavra for palavra in normalizar_tokens(stopwords.words(nome))
        if palavra not in negacoes and not palavra.endswith("n't")
    }
    for codigo, nome in idiomas_stopwords.items()
}


def remover_stopwords(tokens, idioma):
    palavras = stopwords_por_idioma.get(idioma, set())
    return [token for token in tokens if token not in palavras]


for dados, texto, nome_arquivo in [
    (jogos_limpo, "description", "jogos_steam.pkl.gz"),
    (reviews_limpo, "review", "steam_reviews.pkl.gz"),
]:
    coluna_idioma = f"{texto}_idioma"
    coluna_entrada = f"{texto}_tokens_normalizados"
    coluna_saida = f"{texto}_tokens_sem_stopwords"
    if not dados[coluna_idioma].eq('en').all():
        raise ValueError('Execute o filtro de ingles antes de remover stopwords.')
    dados[coluna_saida] = [
        remover_stopwords(tokens, idioma)
        for tokens, idioma in zip(dados[coluna_entrada], dados[coluna_idioma])
    ]

    colunas_listas = [f"{texto}_tokens", coluna_entrada, coluna_saida]
    salvar_pickle(dados, pasta_limpos / nome_arquivo)

    removidos = dados[coluna_entrada].map(len).sum() - dados[coluna_saida].map(len).sum()
    sem_lista = ~dados[coluna_idioma].isin(stopwords_por_idioma)
    print(f"{texto}: {removidos:,} stopwords removidas")
    print(f"{texto}: {sem_lista.sum():,} textos mantidos sem filtragem por idioma")
    display(dados[["appid", coluna_idioma, coluna_entrada, coluna_saida]].head(3))


description: 2,518,134 stopwords removidas
description: 0 textos mantidos sem filtragem por idioma


,appid,description_idioma,description_tokens_normalizados,description_tokens_sem_stopwords
0,10,en,"[play, the, world, s, number, 1, online, actio...","[play, world, number, 1, online, action, game,..."
1,20,en,"[one, of, the, most, popular, online, action, ...","[one, popular, online, action, games, time, te..."
2,30,en,"[enlist, in, an, intense, brand, of, axis, vs,...","[enlist, intense, brand, axis, vs, allied, tea..."


review: 20,234,853 stopwords removidas
review: 0 textos mantidos sem filtragem por idioma


,appid,review_idioma,review_tokens_normalizados,review_tokens_sem_stopwords
0,10,en,"[good, game, ,, most, perfect, counter, strike...","[good, game, ,, perfect, counter, strike, ever..."
2,10,en,"[this, game, is, good, for, anyone, who, wants...","[game, good, anyone, wants, smooth, ,, casual,..."
8,10,en,"[very, good, game, ., i, like, the, older, gra...","[good, game, ., like, older, graphics, gamepla..."


## 8. Alternativa A: stemming

Parte de cópias dos dados apàs a remoção de stopwords e aplica `SnowballStemmer` em ingles. Stemming produz radicais, que podem não ser palavras completas. Negações, tokens não alfabéticos e idiomas sem stemmer permanecem intactos.

Acrescenta colunas com sufixo `_tokens_stemming` e salva os dois arquivos pickle em `_DadosLimpos/stemming/`. Esta alternativa não substitui os DataFrames da etapa principal.


In [25]:
import json
from functools import lru_cache
from nltk.stem.snowball import SnowballStemmer

stemmers = {
    codigo: SnowballStemmer(nome)
    for codigo, nome in idiomas_stopwords.items()
    if nome in SnowballStemmer.languages
}


@lru_cache(maxsize=100000)
def aplicar_stemming(token, idioma):
    stemmer = stemmers.get(idioma)
    if stemmer is None or token in negacoes or not token.isalpha():
        return token
    return sys.intern(stemmer.stem(token))


# As duas alternativas partem dos tokens sem stopwords.
jogos_stemming = jogos_limpo.copy()
reviews_stemming = reviews_limpo.copy()
pasta_stemming = pasta_limpos / "stemming"
pasta_stemming.mkdir(parents=True, exist_ok=True)

for dados, texto, nome_arquivo in [
    (jogos_stemming, "description", "jogos_steam.pkl.gz"),
    (reviews_stemming, "review", "steam_reviews.pkl.gz"),
]:
    entrada = f"{texto}_tokens_sem_stopwords"
    saida = f"{texto}_tokens_stemming"
    idiomas = dados[f"{texto}_idioma"]
    dados[saida] = [
        [aplicar_stemming(token, idioma) for token in tokens]
        for tokens, idioma in zip(dados[entrada], idiomas)
    ]
    colunas_listas = [
        f"{texto}_tokens", f"{texto}_tokens_normalizados", entrada, saida,
    ]
    salvar_pickle(dados, pasta_stemming / nome_arquivo)
    print(f"{texto}: {len(dados):,} registros salvos em {pasta_stemming / nome_arquivo}")
    print(f"Idioma sem stemmer (tokens mantidos): {(~idiomas.isin(stemmers)).sum():,}")
    display(dados[["appid", f"{texto}_idioma", entrada, saida]].head(3))


description: 178,720 registros salvos em C:\Users\berna\Documents\GitHub\PLN_SteamRecommend\_DadosLimpos\stemming\jogos_steam.pkl.gz
Idioma sem stemmer (tokens mantidos): 0


,appid,description_idioma,description_tokens_sem_stopwords,description_tokens_stemming
0,10,en,"[play, world, number, 1, online, action, game,...","[play, world, number, 1, onlin, action, game, ..."
1,20,en,"[one, popular, online, action, games, time, te...","[one, popular, onlin, action, game, time, team..."
2,30,en,"[enlist, intense, brand, axis, vs, allied, tea...","[enlist, intens, brand, axi, vs, alli, teampla..."


review: 379,916 registros salvos em C:\Users\berna\Documents\GitHub\PLN_SteamRecommend\_DadosLimpos\stemming\steam_reviews.pkl.gz
Idioma sem stemmer (tokens mantidos): 0


,appid,review_idioma,review_tokens_sem_stopwords,review_tokens_stemming
0,10,en,"[good, game, ,, perfect, counter, strike, ever...","[good, game, ,, perfect, counter, strike, ever..."
2,10,en,"[game, good, anyone, wants, smooth, ,, casual,...","[game, good, anyon, want, smooth, ,, casual, c..."
8,10,en,"[good, game, ., like, older, graphics, gamepla...","[good, game, ., like, older, graphic, gameplay..."


## 9. Alternativa B: lematização

Parte dos mesmos tokens sem stopwords usados no stemming, sem aplicar lematização aos radicais. Usa `simplemma` para obter lemas por dicionário em ingles; não resolve ambiguidades pelo contexto da frase.

Negações, tokens não alfabéticos e idiomas não suportados permanecem intactos. Acrescenta colunas com sufixo `_tokens_lematizados` e salva os dois arquivos pickle em `_DadosLimpos/lematizacao/`.


In [26]:
import json
from functools import lru_cache
import simplemma

idiomas_lematizacao = {"en"}


@lru_cache(maxsize=100000)
def lematizar_token(token, idioma):
    if idioma not in idiomas_lematizacao or token in negacoes or not token.isalpha():
        return token
    # Lematizacao por dicionario: nao desambigua pelo contexto da frase.
    return sys.intern(simplemma.lemmatize(token, lang=idioma))


jogos_lematizados = jogos_limpo.copy()
reviews_lematizadas = reviews_limpo.copy()
pasta_lematizacao = pasta_limpos / "lematizacao"
pasta_lematizacao.mkdir(parents=True, exist_ok=True)

for dados, texto, nome_arquivo in [
    (jogos_lematizados, "description", "jogos_steam.pkl.gz"),
    (reviews_lematizadas, "review", "steam_reviews.pkl.gz"),
]:
    entrada = f"{texto}_tokens_sem_stopwords"
    saida = f"{texto}_tokens_lematizados"
    idiomas = dados[f"{texto}_idioma"]
    dados[saida] = [
        [lematizar_token(token, idioma) for token in tokens]
        for tokens, idioma in zip(dados[entrada], idiomas)
    ]
    colunas_listas = [
        f"{texto}_tokens", f"{texto}_tokens_normalizados", entrada, saida,
    ]
    salvar_pickle(dados, pasta_lematizacao / nome_arquivo)
    print(f"{texto}: {len(dados):,} registros salvos em {pasta_lematizacao / nome_arquivo}")
    print(f"Idioma sem lematizador (tokens mantidos): {(~idiomas.isin(idiomas_lematizacao)).sum():,}")
    display(dados[["appid", f"{texto}_idioma", entrada, saida]].head(3))


description: 178,720 registros salvos em C:\Users\berna\Documents\GitHub\PLN_SteamRecommend\_DadosLimpos\lematizacao\jogos_steam.pkl.gz
Idioma sem lematizador (tokens mantidos): 0


,appid,description_idioma,description_tokens_sem_stopwords,description_tokens_lematizados
0,10,en,"[play, world, number, 1, online, action, game,...","[play, world, number, 1, online, action, game,..."
1,20,en,"[one, popular, online, action, games, time, te...","[one, popular, online, action, game, time, tea..."
2,30,en,"[enlist, intense, brand, axis, vs, allied, tea...","[enlist, intense, brand, axis, versus, ally, t..."


review: 379,916 registros salvos em C:\Users\berna\Documents\GitHub\PLN_SteamRecommend\_DadosLimpos\lematizacao\steam_reviews.pkl.gz
Idioma sem lematizador (tokens mantidos): 0


,appid,review_idioma,review_tokens_sem_stopwords,review_tokens_lematizados
0,10,en,"[good, game, ,, perfect, counter, strike, ever...","[good, game, ,, perfect, counter, strike, ever..."
2,10,en,"[game, good, anyone, wants, smooth, ,, casual,...","[game, good, anyone, want, smooth, ,, casual, ..."
8,10,en,"[good, game, ., like, older, graphics, gamepla...","[good, game, ., like, old, graphic, gameplay, ..."


## 10. Comparação das etapas

Seleciona até 10 jogos mantidos pela limpeza e a primeira avaliação disponível de cada um. Jogos sem avaliação continuam na amostra. Mostra o texto original, a limpeza, o idioma, os tokens e as duas alternativas finais.

Esta célula é demonstrativa e não grava arquivos. A exibição limita textos e listas para facilitar a leitura; os resultados completos ficam nos DataFrames. Use a comparação para inspecionar transformações, sem tratá-la como avaliação quantitativa da qualidade do pipeline. As saídas já armazenadas no notebook podem corresponder a execuções anteriores.


In [27]:
from IPython.display import display

# Seleciona ate 10 jogos mantidos e a primeira review disponivel de cada um.
ids_pipeline = jogos_limpo["appid"].drop_duplicates().head(10)
amostra_jogos = jogos.loc[
    jogos["appid"].isin(ids_pipeline), ["appid", "name", "description"]
].drop_duplicates("appid")
amostra_reviews = (
    reviews.loc[reviews_limpo.index[reviews_limpo["appid"].isin(ids_pipeline)], ["appid", "review"]]
    .drop_duplicates("appid", keep="first")
    .assign(tem_review=True)
)
amostra_pipeline = (
    amostra_jogos
    .merge(amostra_reviews, on="appid", how="left", validate="one_to_one")
    .assign(tem_review=lambda df: df["tem_review"].eq(True))
    .rename(columns={"description": "descricao_original", "review": "review_original"})
)


def etapa_filtros(df):
    return df.assign(
        descricao_valida=~(
            df["descricao_original"].isna()
            | df["descricao_original"].astype("string").str.strip().eq("")
        ),
        descricao_em_ingles=df["descricao_original"].map(limpar_descricao).map(detectar_idioma).eq("en"),
    )


def etapa_limpeza(df):
    return df.assign(
        descricao_limpa=df['descricao_original'].map(limpar_descricao),
        review_limpa=df['review_original'].map(limpar_texto),
    )


def etapa_textos(df, sufixo_entrada, sufixo_saida, funcao):
    return df.assign(**{
        f"{campo}_{sufixo_saida}": df[f"{campo}_{sufixo_entrada}"].map(funcao)
        for campo in ("descricao", "review")
    })


def etapa_por_idioma(df, sufixo_saida, funcao):
    return df.assign(**{
        f"{campo}_{sufixo_saida}": [
            funcao(tokens, idioma)
            for tokens, idioma in zip(
                df[f"{campo}_normalizados"], df[f"{campo}_idioma"]
            )
        ]
        for campo in ("descricao", "review")
    })


def etapa_alternativa(df, sufixo_saida, funcao):
    return df.assign(**{
        f"{campo}_{sufixo_saida}": [
            [funcao(token, idioma) for token in tokens]
            for tokens, idioma in zip(
                df[f"{campo}_sem_stopwords"], df[f"{campo}_idioma"]
            )
        ]
        for campo in ("descricao", "review")
    })


resultado_pipeline = (
    amostra_pipeline
    .pipe(etapa_filtros)
    .pipe(etapa_limpeza)
    .pipe(etapa_textos, "limpa", "idioma", detectar_idioma)
    .pipe(etapa_textos, "limpa", "tokens", tokenizar_texto)
    .pipe(etapa_textos, "tokens", "normalizados", normalizar_tokens)
    .pipe(etapa_por_idioma, "sem_stopwords", remover_stopwords)
    .pipe(etapa_alternativa, "stemming", aplicar_stemming)
    .pipe(etapa_alternativa, "lematizados", lematizar_token)
)

etapas_pipeline = {
    "Texto original": "original",
    "Limpeza": "limpa",
    "Idioma estimado": "idioma",
    "Tokenizacao": "tokens",
    "Normalizacao": "normalizados",
    "Sem stopwords": "sem_stopwords",
    "Stemming (alternativa)": "stemming",
    "Lematizacao (alternativa)": "lematizados",
}
comparacao_pipeline = pd.concat([
    resultado_pipeline[
        ["appid", "name", f"descricao_{sufixo}", f"review_{sufixo}"]
    ].rename(columns={
        f"descricao_{sufixo}": "descricao",
        f"review_{sufixo}": "review",
    }).assign(etapa=etapa)
    for etapa, sufixo in etapas_pipeline.items()
], ignore_index=True)

print(f"Pipeline demonstrativo: {len(resultado_pipeline)} jogos mantidos pela limpeza.")
print("Reviews: primeira review em ingles mantida de cada jogo; jogos sem review permanecem na amostra.")
print("Previa limitada a 160 caracteres/20 tokens; resultados completos nos DataFrames.")
with pd.option_context("display.max_colwidth", 160, "display.max_seq_items", 20):
    display(resultado_pipeline[
        ["appid", "name", "descricao_valida", "descricao_em_ingles", "tem_review"]
    ])
    for appid, nome in resultado_pipeline[["appid", "name"]].itertuples(index=False, name=None):
        print(f"{appid} - {nome}")
        display(
            comparacao_pipeline.loc[comparacao_pipeline["appid"].eq(appid)]
            .set_index("etapa")[["descricao", "review"]]
        )


Pipeline demonstrativo: 10 jogos mantidos pela limpeza.
Reviews: primeira review em ingles mantida de cada jogo; jogos sem review permanecem na amostra.
Previa limitada a 160 caracteres/20 tokens; resultados completos nos DataFrames.


,appid,name,descricao_valida,descricao_em_ingles,tem_review
0,10,Counter-Strike,True,True,True
1,20,Team Fortress Classic,True,True,True
2,30,Day of Defeat,True,True,True
3,40,Deathmatch Classic,True,True,True
4,50,Half-Life: Opposing Force,True,True,True
5,60,Ricochet,True,True,True
6,70,Half-Life,True,True,True
7,80,Counter-Strike: Condition Zero,True,True,True
8,130,Half-Life: Blue Shift,True,True,True
9,220,Half-Life 2,True,True,True


10 - Counter-Strike


,descricao,review
etapa,,
Texto original,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with...,"Good game, most perfect counter strike ever existed the only problem is this version does not have bots only Condition Zero has bots"
Limpeza,Play the world s number 1 online action game Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team based game Ally with t...,"Good game, most perfect counter strike ever existed the only problem is this version does not have bots only Condition Zero has bots"
Idioma estimado,en,en
Tokenizacao,"[Play, the, world, s, number, 1, online, action, game, Engage, in, an, incredibly, realistic, brand, of, terrorist, warfare, in, this, ...]","[Good, game, ,, most, perfect, counter, strike, ever, existed, the, only, problem, is, this, version, does, not, have, bots, only, ...]"
Normalizacao,"[play, the, world, s, number, 1, online, action, game, engage, in, an, incredibly, realistic, brand, of, terrorist, warfare, in, this, ...]","[good, game, ,, most, perfect, counter, strike, ever, existed, the, only, problem, is, this, version, does, not, have, bots, only, ...]"
Sem stopwords,"[play, world, number, 1, online, action, game, engage, incredibly, realistic, brand, terrorist, warfare, wildly, popular, team, based, game, ally, teammates...","[good, game, ,, perfect, counter, strike, ever, existed, problem, version, not, bots, condition, zero, bots]"
Stemming (alternativa),"[play, world, number, 1, onlin, action, game, engag, incred, realist, brand, terrorist, warfar, wild, popular, team, base, game, alli, teammat, ...]","[good, game, ,, perfect, counter, strike, ever, exist, problem, version, not, bot, condit, zero, bot]"
Lematizacao (alternativa),"[play, world, number, 1, online, action, game, engage, incredibly, realistic, brand, terrorist, warfare, wildly, popular, team, base, game, ally, teammate, ...","[good, game, ,, perfect, counter, strike, ever, exist, problem, version, not, bot, condition, zero, bot]"


20 - Team Fortress Classic


...

,descricao,review
etapa,,
Texto original,"With its extensive Tour of Duty campaign, a near-limitless number of skirmish modes, updates and new content for Counter-Strike's award-winning multiplayer ...","this version of CS really is the least popular but known version of the franchise, this sh1t was so useless because not only is it just a 1.6 remake with re..."
Limpeza,With its extensive Tour of Duty campaign a near limitless number of skirmish modes updates and new content for Counter Strike s award winning multiplayer ga...,"this version of CS really is the least popular but known version of the franchise, this sh1t was so useless because not only is it just a 1.6 remake with re..."
Idioma estimado,en,en
Tokenizacao,"[With, its, extensive, Tour, of, Duty, campaign, a, near, limitless, number, of, skirmish, modes, updates, and, new, content, for, Counter, ...]","[this, version, of, CS, really, is, the, least, popular, but, known, version, of, the, franchise, ,, this, sh1t, was, so, ...]"
Normalizacao,"[with, its, extensive, tour, of, duty, campaign, a, near, limitless, number, of, skirmish, modes, updates, and, new, content, for, counter, ...]","[this, version, of, cs, really, is, the, least, popular, but, known, version, of, the, franchise, ,, this, sh1t, was, so, ...]"
Sem stopwords,"[extensive, tour, duty, campaign, near, limitless, number, skirmish, modes, updates, new, content, counter, strike, award, winning, multiplayer, game, play,...","[version, cs, really, least, popular, known, version, franchise, ,, sh1t, useless, not, 1.6, remake, revamped, textures, also, still, runs, old, ...]"
Stemming (alternativa),"[extens, tour, duti, campaign, near, limitless, number, skirmish, mode, updat, new, content, counter, strike, award, win, multiplay, game, play, plus, ...]","[version, cs, realli, least, popular, known, version, franchis, ,, sh1t, useless, not, 1.6, remak, revamp, textur, also, still, run, old, ...]"
Lematizacao (alternativa),"[extensive, tour, duty, campaign, near, limitless, number, skirmish, mode, update, new, content, counter, strike, award, win, multiplayer, game, play, plus,...","[version, c, really, less, popular, know, version, franchise, ,, sh1t, useless, not, 1.6, remake, revamp, texture, also, still, run, old, ...]"


130 - Half-Life: Blue Shift


,descricao,review
etapa,,
Texto original,"Made by Gearbox Software and originally released in 2001 as an add-on to Half-Life, Blue Shift is a return to the Black Mesa Research Facility in which you ...",sucks id say?? its good but it were more security themed with different weapons id come back to it
Limpeza,Made by Gearbox Software and originally released in 2001 as an add on to Half Life Blue Shift is a return to the Black Mesa Research Facility in which you p...,sucks id say?? its good but it were more security themed with different weapons id come back to it
Idioma estimado,en,en
Tokenizacao,"[Made, by, Gearbox, Software, and, originally, released, in, 2001, as, an, add, on, to, Half, Life, Blue, Shift, is, a, ...]","[sucks, id, say, ?, ?, its, good, but, it, were, more, security, themed, with, different, weapons, id, come, back, to, ...]"
Normalizacao,"[made, by, gearbox, software, and, originally, released, in, 2001, as, an, add, on, to, half, life, blue, shift, is, a, ...]","[sucks, id, say, ?, ?, its, good, but, it, were, more, security, themed, with, different, weapons, id, come, back, to, ...]"
Sem stopwords,"[made, gearbox, software, originally, released, 2001, add, half, life, blue, shift, return, black, mesa, research, facility, play, barney, calhoun, security...","[sucks, id, say, ?, ?, good, security, themed, different, weapons, id, come, back]"
Stemming (alternativa),"[made, gearbox, softwar, origin, releas, 2001, add, half, life, blue, shift, return, black, mesa, research, facil, play, barney, calhoun, secur, ...]","[suck, id, say, ?, ?, good, secur, theme, differ, weapon, id, come, back]"
Lematizacao (alternativa),"[make, gearbox, software, originally, release, 2001, add, half, life, blue, shift, return, black, mesa, research, facility, play, barney, calhoun, security,...","[suck, id, say, ?, ?, good, security, theme, different, weapon, id, come, back]"


220 - Half-Life 2


,descricao,review
etapa,,
Texto original,"Reawakened from stasis in the occupied metropolis of City 17, Gordon Freeman is joined by Alyx Vance as he leads a desperate human resistance. Experience th...","Best game forever. Valve, please count to three, i am wait half life 3! Thanks to all valve command for this game and half life university"
Limpeza,Reawakened from stasis in the occupied metropolis of City 17 Gordon Freeman is joined by Alyx Vance as he leads a desperate human resistance Experience the ...,"Best game forever. Valve, please count to three, i am wait half life 3! Thanks to all valve command for this game and half life university"
Idioma estimado,en,en
Tokenizacao,"[Reawakened, from, stasis, in, the, occupied, metropolis, of, City, 17, Gordon, Freeman, is, joined, by, Alyx, Vance, as, he, leads, ...]","[Best, game, forever, ., Valve, ,, please, count, to, three, ,, i, am, wait, half, life, 3, !, Thanks, to, ...]"
Normalizacao,"[reawakened, from, stasis, in, the, occupied, metropolis, of, city, 17, gordon, freeman, is, joined, by, alyx, vance, as, he, leads, ...]","[best, game, forever, ., valve, ,, please, count, to, three, ,, i, am, wait, half, life, 3, !, thanks, to, ...]"
Sem stopwords,"[reawakened, stasis, occupied, metropolis, city, 17, gordon, freeman, joined, alyx, vance, leads, desperate, human, resistance, experience, landmark, first,...","[best, game, forever, ., valve, ,, please, count, three, ,, wait, half, life, 3, !, thanks, valve, command, game, half, ...]"
Stemming (alternativa),"[reawaken, stasi, occupi, metropoli, citi, 17, gordon, freeman, join, alyx, vanc, lead, desper, human, resist, experi, landmark, first, person, shooter, ...]","[best, game, forev, ., valv, ,, pleas, count, three, ,, wait, half, life, 3, !, thank, valv, command, game, half, ...]"
Lematizacao (alternativa),"[reawaken, stasis, occupy, metropolis, city, 17, Gordon, freeman, join, alyx, vance, lead, desperate, human, resistance, experience, landmark, first, person...","[good, game, forever, ., valve, ,, please, count, three, ,, wait, half, life, 3, !, thanks, valve, command, game, half, ...]"
